# Make Models

B Steele, Colorado State University, ROSSyndicate — last updated Sep 17, 2026

This notebook trains the production Secchi-depth model: an ensemble of XGBoost models over ten independent HUC8-random CV arrangements built in `03_split_data.ipynb`, evaluated on that notebook's one fixed, never-trained-on holdout.

**Single algorithm, ten seeds.** An earlier version of this pipeline ensembled 5 seeds × 2 algorithms (XGBoost + LightGBM), for 40 models total. A variance decomposition of that ensemble's holdout predictions found the two algorithms agreed very closely with one another (0.998 correlation between their respective 5-seed ensemble means) while different random seeds disagreed with each other substantially more (~0.99 pairwise correlation); seed choice accounted for roughly 65% of the ensemble's internal prediction spread, versus roughly 13% for model architecture. Given that seed, not algorithm, was the dominant source of useful ensemble variability, this pipeline runs a single algorithm (XGBoost) over 10 independent random seeds instead (`601`–`610`) — 10 seeds × 4 folds = 40 models, the same total model count as the original two-algorithm design, for no measurable accuracy cost and a simpler codebase to maintain.

Each ensemble seed runs its own complete feature-selection + tuning + training pipeline, entirely independently, on its own 4-fold split of the shared CV pool. The final production prediction is the average across all 10 seeds × 4 folds.</cell id="4dbf7522">


## Setup

In [1]:
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("python").resolve()))
import backward_elim as be
import features
import metrics as M
import model_xgb
import spatial_cv
from tuning import gap_aware_tune

be.register_model("xgboost", model_xgb)

AQUAMATCH_DIR = Path("aquamatch_files")
MODELS_DIR = Path("xg_models") / "v3_production"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "harmonized_value"
MODEL_NAME = "xgboost"
MOD = model_xgb
SEEDS = list(range(601, 611))
N_TRIALS_INITIAL = 25
N_TRIALS_FINAL = 40
ELIM_TOL = 0.005
ELIM_SEED = 47
TUNE_SEED = 47

SITE_COLS = ["elevation_m", "catchment_area_sqkm", "pct_impervious_2006", "pct_urban_2006",
             "pct_forest_2006", "pct_cropland_2006", "pct_wetland_2006", "shore_flag"]
WEATHER_COLS_CURATED = ["precip_mm_prev30", "tmax_degC_prev30"]


def log(seed, msg):
    print(f"[{time.strftime('%H:%M:%S')}][seed{seed}] {msg}", flush=True)

## Load data and build 10 fresh CV-fold arrangements

Reads `03_split_data.ipynb`'s output (the same fixed holdout used everywhere else in this project) and adds 10 new `cvfold_seed{601-610}` columns via `spatial_cv.build_splits` — the holdout itself is rebuilt deterministically from the same `holdout_seed=3`, so it comes back identical to `03`'s; only the CV-pool fold arrangement is new. `04`'s original `cvfold_seed{501-505}` columns are left untouched alongside these.

`features.coarsen_site_features()` reduces `catchment_area_sqkm` (log-scale binned) and the `pct_*_2006` land-cover fractions (rounded to the nearest percentage point) so a heavily-resampled single waterbody's static, per-comid LakeCat values can't act as a de facto waterbody ID for the model to key a memorized SDD value off of. It does not (and isn't meant to) address the separate, already-documented leakage from the handful of waterbodies spanning more than one HUC8 (Lake Powell chief among them) — those stay large and distinct enough to remain identifiable after coarsening by design.

In [2]:
df = pd.read_parquet(AQUAMATCH_DIR / "base_with_splits.parquet")
df = features.add_spectral_indices(df)
df = features.coarsen_site_features(df)
df = spatial_cv.build_splits(df, ensemble_seeds=SEEDS)
print(f"{len(df):,} rows, {df.shape[1]} columns; holdout n={df['is_holdout'].sum():,}")

11,143 rows, 72 columns; holdout n=2,334


## Per-seed pipeline (single algorithm)

Same stages as `04_make_models.ipynb`, minus the cross-algorithm joint elimination step (not applicable with one algorithm) — the single model's own backward-elimination result is the seed's final feature set directly, with no intersection step.

In [3]:
def run_seed_pipeline(seed, df):
    out_dir = MODELS_DIR / f"seed{seed}"
    out_dir.mkdir(parents=True, exist_ok=True)
    t_start = time.time()

    folds = spatial_cv.build_folds(df, seed)
    test = df[df["is_holdout"]].reset_index(drop=True)
    log(seed, f"folds: {[(f.part, len(f.train), len(f.val)) for f in folds]}, holdout n={len(test)}")

    # --- candidate pool + correlation pruning (this seed's own train_val pool) ---
    metadata_cols = {"HUC8", "holdout_part", "is_holdout"} | {f"cvfold_seed{s}" for s in SEEDS}
    extra_cols = set(SITE_COLS) | set(WEATHER_COLS_CURATED) | metadata_cols
    optical_feats = [f for f in features.candidate_feature_list(df) if f not in extra_cols]
    candidates = optical_feats + SITE_COLS + WEATHER_COLS_CURATED

    train_val = pd.concat([f.train for f in folds] + [folds[0].val], ignore_index=True).drop_duplicates(
        subset=["siteSR_id", "date"])
    pruned = features.correlation_prune(train_val, candidates, target=TARGET)
    log(seed, f"candidate pool: {len(candidates)} -> {len(pruned)} after correlation pruning")

    # tuning seed is derived per-ensemble-seed (rather than a single fixed value shared
    # by every seed) so each seed's random search explores a genuinely different draw of
    # hyperparameter combinations - previously every seed reused seed=47 verbatim, so all
    # 10 seeds evaluated the identical set of combinations against their own data instead
    # of diversifying search coverage across the ensemble the way the CV folds and feature
    # selection already do
    seed_tune_seed = TUNE_SEED + seed

    # --- stage 1: initial gap-aware tuning (fixed params for elimination) ---
    log(seed, f"=== initial gap-aware tuning on {len(pruned)}-feature pool ===")
    init_result = gap_aware_tune(MOD, folds, pruned, TARGET, n_trials=N_TRIALS_INITIAL, seed=seed_tune_seed)
    fixed_params = init_result["best_params"]
    log(seed, f"val_rmse={init_result['best_score']:.4f} gap={init_result['best_gap']:.4f}")

    # --- stage 2: backward elimination (single algorithm, no joint step) ---
    log(seed, "=== backward elimination ===")
    elim_result = be.backward_eliminate(MODEL_NAME, folds, TARGET, fixed_params, pruned,
                                        weight_fn=None, tol=ELIM_TOL, seed=ELIM_SEED,
                                        log=lambda m: log(seed, m))
    final_feats = elim_result["final_features"]
    log(seed, f"final ({len(final_feats)}): {final_feats}")

    with open(out_dir / "backward_elim_summary.json", "w") as fh:
        json.dump(dict(starting_candidates=candidates, correlation_pruned=pruned,
                       final_features=final_feats, path=elim_result["path"]), fh, indent=2, default=str)

    # --- stage 3: final gap-aware tuning on the final feature set ---
    log(seed, f"=== final gap-aware tuning on {len(final_feats)}-feature set ===")
    tune_result = gap_aware_tune(MOD, folds, final_feats, TARGET, n_trials=N_TRIALS_FINAL, seed=seed_tune_seed)
    log(seed, f"chosen: val_rmse={tune_result['best_score']:.4f} gap={tune_result['best_gap']:.4f}")
    with open(out_dir / f"final_tune_{MODEL_NAME}.json", "w") as fh:
        json.dump({"model": MODEL_NAME, "features": final_feats, **tune_result}, fh, indent=2, default=str)

    # --- stage 4: final unweighted training + evaluation ---
    params = tune_result["best_params"]
    fold_models = MOD.train_fold_models(folds, final_feats, TARGET, params)
    for i, fm in enumerate(fold_models):
        fm.save_model(str(out_dir / f"{MODEL_NAME}_fold{i+1}.json"))

    cv_preds, cv_ys = [], []
    for f, model in zip(folds, fold_models):
        cv_preds.append(MOD.predict_ensemble([model], f.val[final_feats]))
        cv_ys.append(f.val[TARGET].values)
    cv_metrics = M.all_metrics(np.concatenate(cv_ys), np.concatenate(cv_preds))

    test_pred = MOD.predict_ensemble(fold_models, test[final_feats])
    test_metrics = M.all_metrics(test[TARGET].values, test_pred)
    log(seed, f"FINAL: cv_rmse={cv_metrics['rmse']:.4f} holdout_rmse={test_metrics['rmse']:.4f}")

    summary = {MODEL_NAME: dict(features=final_feats, params=params, cv_metrics=cv_metrics, test_metrics=test_metrics)}
    with open(out_dir / "final_eval_summary.json", "w") as fh:
        json.dump(summary, fh, indent=2, default=str)

    r = test[["siteSR_id", "date", "HUC4", "HUC8"]].copy()
    r["seed"] = seed
    r["model"] = MODEL_NAME
    r["y"] = test[TARGET].values
    r["pred"] = test_pred
    r.to_parquet(out_dir / "holdout_predictions.parquet")

    log(seed, f"SEED {seed} DONE in {time.time()-t_start:.0f}s")
    return summary

In [4]:
all_summaries = {}
for seed in SEEDS:
    all_summaries[seed] = run_seed_pipeline(seed, df)

[22:56:31][seed601] folds: [(1, 6720, 2089), (2, 6535, 2274), (3, 6431, 2378), (4, 6741, 2068)], holdout n=2334


[22:56:31][seed601] candidate pool: 45 -> 34 after correlation pruning


[22:56:31][seed601] === initial gap-aware tuning on 34-feature pool ===


[23:00:36][seed601] val_rmse=1.3229 gap=0.3833


[23:00:36][seed601] === backward elimination ===


...

[01:50:13][seed610] final (32): ['atm_corr_LaSRC', 'red_corr7', 'nir_corr7', 'swir1_corr7', 'temp_corr7', 'precip_mm_prev1', 'tmean_degC_prev7', 'srad_Wm2_prev1', 'srad_Wm2_prev7', 'precip_mm_prev7', 'tmax_degC_prev7', 'tmin_degC_prev7', 'tmean_degC_prev30', 'tmin_degC_prev30', 'srad_Wm2_prev30', 'BR', 'BG', 'NR', 'fai', 'NDVI', 'NDSSI', 'NDWI', 'MNDWI', 'elevation_m', 'catchment_area_sqkm', 'pct_impervious_2006', 'pct_urban_2006', 'pct_forest_2006', 'pct_cropland_2006', 'pct_wetland_2006', 'shore_flag', 'tmax_degC_prev30']


[01:50:13][seed610] === final gap-aware tuning on 32-feature set ===


[01:56:32][seed610] chosen: val_rmse=1.3443 gap=0.4503


[01:56:50][seed610] FINAL: cv_rmse=1.3499 holdout_rmse=1.8798


[01:56:50][seed610] SEED 610 DONE in 825s


## Per-seed summary

In [5]:
rows = []
for seed, summary in all_summaries.items():
    s = summary[MODEL_NAME]
    rows.append(dict(seed=seed, n_features=len(s["features"]), cv_rmse=s["cv_metrics"]["rmse"],
                     holdout_rmse=s["test_metrics"]["rmse"], holdout_bias=s["test_metrics"]["bias"]))
pd.DataFrame(rows).sort_values("seed")

,seed,n_features,cv_rmse,holdout_rmse,holdout_bias
0,601,34,1.334909,1.866912,0.261629
1,602,25,1.319448,1.856715,0.276971
2,603,26,1.346800,1.868308,0.331983
3,604,25,1.317395,1.814210,0.127317
4,605,30,1.372884,1.821035,0.233165
5,606,28,1.333762,1.870621,0.270614
6,607,27,1.340205,1.900975,0.265158
7,608,30,1.352509,1.851494,0.284390
8,609,28,1.306364,1.826814,0.163618
9,610,32,1.349910,1.879784,0.300789


## Assemble the 10-seed ensemble

40 models (10 seeds x 4 folds). Full evaluation (HUC8 breakdown, seed-vs-seed spread, SHAP) happens in `05_evaluate_ensemble.ipynb`; this is a quick sanity check on the assembled holdout metrics before moving on.

In [6]:
rows = []
for seed in SEEDS:
    rows.append(pd.read_parquet(MODELS_DIR / f"seed{seed}" / "holdout_predictions.parquet"))
all_preds = pd.concat(rows, ignore_index=True)

ensemble_pred = all_preds.groupby(["siteSR_id", "date"]).agg(y=("y", "first"), pred=("pred", "mean")).reset_index()
ensemble_metrics = M.all_metrics(ensemble_pred["y"], ensemble_pred["pred"])
print(f"production ensemble (10 seeds x xgboost, 40 models): {ensemble_metrics}")

production ensemble (10 seeds x xgboost, 40 models): {'rmse': 1.8466498435492145, 'mae': 1.3469278292199338, 'mape': 0.6974475150947105, 'bias': 0.21648619859614288, 'p_bias': 5.404338563371457, 'smape': 0.38109833864420484, 'r2': 0.5075473903548398, 'n': 2133}


## Summary

The production model is a 40-member ensemble — 10 independent HUC8-random CV arrangements x 4 folds each — trained unweighted, with gap-aware hyperparameter selection at every tuning stage and per-seed feature sets chosen by correlation pruning followed by shadow-decoy backward elimination. Every member is evaluated on one fixed HUC8-random holdout that no model ever trains on. Artifacts (fold models, per-seed feature/hyperparameter configs, holdout predictions) are saved under `xg_models/v3_production/seed{601–610}/`. See `05_evaluate_ensemble.ipynb` for the full evaluation (HUC8 breakdown, seed-vs-seed spread, SHAP).